# Algoritmos - Actividad Guiada 1

**Nombre:** Juan Carlos Vereda Ruiz


## 1) Torres de Hanoi (Divide y vencerás)

Implementación recursiva que devuelve la lista de movimientos y un ejemplo para `N=3`.

In [1]:
def torres_hanoi(n:int, origen:int, destino:int):
    """Devuelve la lista de movimientos (origen, destino) para resolver Hanoi con n discos."""
    if n <= 0:
        return []
    if n == 1:
        return [(origen, destino)]
    auxiliar = 6 - origen - destino  # suponiendo torres 1,2,3
    return (torres_hanoi(n-1, origen, auxiliar)
            + [(origen, destino)]
            + torres_hanoi(n-1, auxiliar, destino))

movs = torres_hanoi(3, 1, 3)
print("Movimientos:", movs)
print("Número de movimientos:", len(movs))

Movimientos: [(1, 3), (1, 2), (3, 2), (1, 3), (2, 1), (2, 3), (1, 3)]
Número de movimientos: 7


## 2) Sucesión de Fibonacci

Versión recursiva (tal como se suele pedir en la actividad) y versión iterativa eficiente.

In [2]:
def fibonacci_rec(n:int) -> int:
    """Devuelve F(n) con F(0)=1, F(1)=1 (convención usada en el enunciado original)."""
    if n < 2:
        return 1
    return fibonacci_rec(n-1) + fibonacci_rec(n-2)

def fibonacci_it(n:int) -> int:
    a, b = 1, 1
    for _ in range(n):
        a, b = b, a+b
    return a

print("Fibonacci rec F(5):", fibonacci_rec(5))
print("Fibonacci it  F(5):", fibonacci_it(5))

Fibonacci rec F(5): 8
Fibonacci it  F(5): 8


## 3) Devolución de cambio (Técnica voraz)

Dado un importe `N` y un sistema monetario `SM` ordenado de mayor a menor, calcula cuántas monedas de cada tipo usar.

In [3]:
def cambio_monedas(n:int, sm:list[int]) -> list[int] | None:
    """Algoritmo voraz. Devuelve una lista con el número de monedas por denominación.
    Si no puede completar exactamente el cambio, devuelve None."""
    solucion = [0] * len(sm)
    restante = n
    for i, valor in enumerate(sm):
        if valor <= 0:
            raise ValueError("Las denominaciones deben ser positivas.")
        monedas = restante // valor
        solucion[i] = monedas
        restante -= monedas * valor
        if restante == 0:
            return solucion
    return None

print("Cambio 15 con [25,10,5,1] =>", cambio_monedas(15, [25,10,5,1]))

Cambio 15 con [25,10,5,1] => [0, 1, 1, 0]


## 4) N-Reinas (Vuelta atrás / Backtracking)

Se coloca una reina por columna. La solución `S` guarda en `S[col]` la fila (1..N) donde se coloca la reina.

In [6]:
def es_prometedora(sol:list[int], etapa:int) -> bool:
    # misma fila
    if len(set(sol[:etapa+1])) < (etapa+1):
        return False
    # diagonales
    for i in range(etapa+1):
        for j in range(i+1, etapa+1):
            if abs(i-j) == abs(sol[i]-sol[j]):
                return False
    return True

def n_reinas(n:int):
    sol = [0]*n
    soluciones = []

    def bt(etapa:int):
        for fila in range(1, n+1):
            sol[etapa] = fila
            if es_prometedora(sol, etapa):
                if etapa == n-1:
                    soluciones.append(sol.copy())
                else:
                    bt(etapa+1)
            sol[etapa] = 0

    bt(0)
    return soluciones

sols8 = n_reinas(8)
print("Número de soluciones para 8-reinas:", len(sols8))
print("Ejemplo de solución:", sols8[0])

Número de soluciones para 8-reinas: 92
Ejemplo de solución: [1, 5, 8, 6, 3, 7, 2, 4]


## 5) Viaje por el río (Programación dinámica)

Se dispone de una matriz de tarifas `T[i][j]` (coste directo de ir de i a j). Se desea el coste mínimo y reconstruir una ruta óptima.

Al ser un problema con estructura acíclica (i < j), se resuelve con DP: `precio[i][j] = min( T[i][j], min_{k in [i..j-1]} precio[i][k] + T[k][j] )`.

In [7]:
INF = 10**9

TARIFAS = [
    [0, 5, 4, 3, 999, 999, 999],
    [999, 0, 999, 2, 3, 999, 11],
    [999, 999, 0, 1, 999, 4, 10],
    [999, 999, 999, 0, 5, 6, 9],
    [999, 999, 999, 999, 0, 999, 4],
    [999, 999, 999, 999, 999, 0, 3],
    [999, 999, 999, 999, 999, 999, 0],
]

def precios_y_ruta(tarifas:list[list[int]]):
    n = len(tarifas)
    precios = [[INF]*n for _ in range(n)]
    prev = [[None]*n for _ in range(n)]  # prev[i][j] = k (última parada antes de j viniendo desde i)

    for i in range(n):
        precios[i][i] = 0
        prev[i][i] = i

    for i in range(n-1, -1, -1):
        for j in range(i+1, n):
            # opción directa
            mejor = tarifas[i][j]
            mejor_prev = i
            # opción con escala k
            for k in range(i, j):
                cand = precios[i][k] + tarifas[k][j]
                if cand < mejor:
                    mejor = cand
                    mejor_prev = k
            precios[i][j] = mejor
            prev[i][j] = mejor_prev
    return precios, prev

def reconstruir_ruta(prev, i:int, j:int):
    """Reconstruye el camino óptimo i -> j usando prev[i][j] como última parada."""
    if i == j:
        return [i]
    k = prev[i][j]
    if k is None:
        return []
    if k == i:
        return [i, j]
    return reconstruir_ruta(prev, i, k) + [j]

PRECIOS, PREV = precios_y_ruta(TARIFAS)
print("Coste mínimo 0->6:", PRECIOS[0][6])
print("Ruta óptima 0->6:", reconstruir_ruta(PREV, 0, 6))

Coste mínimo 0->6: 11
Ruta óptima 0->6: [0, 2, 5, 6]
